In [ ]:
%pip install anthropic python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, stop_sequences = [], temperature = 1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system is not None:
        params["system"] = system

    response = client.messages.create(**params)
    return response.content[0].text

In [ ]:
import json

def generate_dataset():
    prompt = """
                Generate dataset for ...

                Example output:

                ```json
                    [
                        { 
                            "task": "blah blah blah",
                            "format": "python"
                        },
                        {
                            "task": "more blah blah blah"
                            "format": "json"
                        }
                        ...additional
                    ]
                ```
                * more instructions
                * more and more instructions
            """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [ ]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def grade_by_model(test_case, output):
    """
    Docstring for grade_by_model
    
    :param test_case: Description
    """
    eval_prompt = f"""
                You are a reviewer for an AI generated solution for the given task:
                
                Task: {test_case["Task"]}
                AI Output: {output}

                Your output must be of the following format:

                JSON output
                ```json
                    [
                        {
                            "strengths": string[],
                            "weaknesses": string[],
                            "reasoning": string,
                            "score": number (out of 10)
                        }
                    ]
                ```
            """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    grade = chat(messages, stop_sequences=["```"])
    return json.loads(grade)


In [ ]:
import ast
import re

def validate_format(test_case):
    task = test_case["task"]
    format = test_case["format"]
    if format == "json":
        try:
            json.loads(task.strip())
            return True
        except:
            return False
    elif format == "python":
        try:
            ast.parse(task.strip())
            return True
        except:
            return False
    elif format == "re":
        try:
            re.compile(task.strip())
            return True
        except:
            return False
    else:
        return False

In [ ]:
from numpy import mean

def run_prompt(test_case):
    """
    Merges the prompt and test case input and then returns the result
    
    :param test_case: Description
    """
    prompt = f"""
                Please solve the following task:
                {test_case["task"]}
            """
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

def run_test_case(test_case):
    """
    Calls run_prompt() and grades the result

    :param test_case: Description
    """
    output = run_prompt(test_case)
    
    grade = grade_by_model(test_case, output)

    result = {
        "test_case": test_case,
        "output": output,
        "score": grade["score"],
        "reasoning": grade["reasoning"]
    } 
    return result

def run_eval(dataset):
    """
    Loads the dataset and calls run_test_case() for each test case 
    
    :param dataset: Description
    """

    results = []
    for test_case in dataset:
        results.append(run_test_case(test_case))

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))